# 04 — Análise estatística (nível estadual — 645 municípios)

**Objetivo:** testar a hipótese central do estudo:

> Municípios de SP com maior proporção de idosos morando sozinhos têm maior
> taxa de internação por causas associadas a falta de socorro imediato
> (lesões/causas externas, sintomas mal definidos, transtornos mentais),
> mesmo controlando pelo IDH municipal?

**Entrada:** `data/processed/dataset_municipios_sp.csv` (nível município,
645 linhas, gerado no notebook 03) — **é essa tabela que usamos para testar
a hipótese**, não o painel (`dataset_consolidado_sp.csv`). O painel tem 5
linhas por município (uma por ano) mas `pct_idosos_sozinhos` é a mesma nas
5 — usá-lo na correlação/regressão infla o "n" artificialmente
(pseudorreplicação, ver notebook 03, seção 3.3). O painel entra só na seção
4.5 (evolução por ano), onde isso não é um problema.

**Saídas:** tabelas e figuras em `outputs/tables/` e `outputs/figures/` —
prontas para entrar na seção de Resultados do artigo.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

mun = pd.read_csv(config.DATA_PROCESSED / "dataset_municipios_sp.csv")
print(mun.shape, "(esperado 645)")
mun.head()


## 4.1 Estatística descritiva


In [ ]:
mun.describe(include="all").T


## 4.2 Correlação: % idosos sozinhos × taxa de internação

Um ponto por município (n=645, ou menos se houver `NaN`).


In [ ]:
x = "pct_idosos_sozinhos"
y = "taxa_internacao_100k_domicilios_idosos"

sub = mun.dropna(subset=[x, y])
r, p = stats.pearsonr(sub[x], sub[y])
print(f"Correlação de Pearson (n={len(sub)}): r={r:.3f}, p={p:.4g}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(data=mun, x=x, y=y, ax=ax, scatter_kws={"alpha": 0.5})

rc = mun[mun["municipio"] == config.RIO_CLARO_NOME]
if not rc.empty:
    ax.scatter(rc[x], rc[y], color="red", s=100, zorder=5, label=config.RIO_CLARO_NOME)
    ax.legend()

ax.set_xlabel("% de domicílios com responsável idoso que são unipessoais")
ax.set_ylabel("Internações por 100 mil domicílios com responsável idoso")
ax.set_title("Idosos sozinhos × internações — municípios de SP (2022-2026)")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "correlacao_idosos_sozinhos_internacoes.png", dpi=150)
plt.show()


## 4.3 Regressão controlando por IDH

Modelo: `taxa_internacao_100k_domicilios_idosos ~ pct_idosos_sozinhos + idhm`


In [ ]:
formula = "taxa_internacao_100k_domicilios_idosos ~ pct_idosos_sozinhos + idhm"
modelo = smf.ols(formula, data=mun).fit()
print(modelo.summary())
with open(config.OUTPUTS_TABLES / "regressao_ols.txt", "w") as f:
    f.write(modelo.summary().as_text())


**Como ler este resultado (preencher com os números reais depois de rodar):**
a correlação simples entre "% idosos sozinhos" e "taxa de internação" tende
a vir fraca e não muito significativa isoladamente — mas isso é esperado:
municípios com mais idosos sozinhos tendem a ter IDH mais baixo, e IDH mais
baixo puxa a taxa de internação por outros caminhos, mascarando o efeito
direto do isolamento até controlar por IDH. Se o coeficiente de
`pct_idosos_sozinhos` ficar positivo e significativo *na regressão* mesmo
sem sê-lo na correlação simples, isso é evidência de **confundimento
(suppressor effect)** — um achado legítimo e válido de se discutir no
artigo, mas exige cuidado na redação: não é "a hipótese não se confirma",
é "a hipótese só aparece depois de controlar pelo desenvolvimento
municipal". Também vale checar o sinal do coeficiente de `idhm`: se vier
positivo (mais IDH → mais internação), pode refletir melhor acesso/
registro hospitalar em municípios mais ricos, não pior saúde — discutir
como limitação de interpretação, não ocultar.


## 4.4 Perfil das internações por causa

Lembrando: cada "causa" aqui é um **capítulo da CID-10** (ver
`config.CAUSAS_SIH`), não o subgrupo específico do plano original.


In [ ]:
painel = pd.read_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv")

causas = list(config.CAUSAS_SIH.keys())
totais = painel[causas].sum().rename(index=config.CAUSAS_SIH_LABELS)

fig, ax = plt.subplots(figsize=(7, 5))
totais.sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Total de internações em idosos, 2022-2026 (estado de SP)")
ax.set_title("Perfil das internações em idosos, por capítulo CID-10")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "perfil_causas.png", dpi=150)
plt.show()


## 4.5 Evolução da taxa de internação por ano (painel)

Aqui sim usamos o painel — é uma descrição da série temporal, não um teste
de hipótese com `pct_idosos_sozinhos`, então a pseudorreplicação não se
aplica.


In [ ]:
evolucao = painel.groupby("ano")["taxa_internacao_100k_domicilios_idosos"].agg(["mean", "median"]).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(evolucao["ano"], evolucao["mean"], marker="o", label="Média")
ax.plot(evolucao["ano"], evolucao["median"], marker="o", label="Mediana")
ax.set_xlabel("Ano")
ax.set_ylabel("Internações por 100 mil domicílios com responsável idoso")
ax.set_title("Evolução da taxa de internação — municípios de SP\n(2026 parcial, até julho)")
ax.legend()
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "evolucao_taxa_internacao_sp.png", dpi=150)
plt.show()


## 4.6 Mapa coroplético (opcional — requer `geopandas` e o shapefile de SP)

1. Baixe: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/SP/SP_Municipios_2022.zip
2. Salve (sem descompactar) como `data/external/sp_municipios.zip`

⚠️ O shapefile do IBGE traz `codigo_ibge` — como nossa base usa nome de
município (seção 3), o merge aqui é por nome normalizado também.


In [ ]:
try:
    import geopandas as gpd

    caminho_shp = config.DATA_EXTERNAL / "sp_municipios.zip"
    if caminho_shp.exists():
        gdf = gpd.read_file(f"zip://{caminho_shp}")
        gdf["municipio_norm"] = gdf["NM_MUN"].apply(config.normalizar_municipio)
        mapa = gdf.merge(mun, on="municipio_norm", how="left")

        fig, ax = plt.subplots(figsize=(9, 9))
        mapa.plot(column="taxa_internacao_100k_domicilios_idosos", cmap="OrRd", legend=True, ax=ax,
                  missing_kwds={"color": "lightgrey"})
        ax.set_title("Taxa de internação em idosos — SP (2022-2026)")
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(config.OUTPUTS_FIGURES / "mapa_taxa_internacao_sp.png", dpi=150)
        plt.show()
    else:
        print(f"Baixe o shapefile conforme instruções acima e salve em {caminho_shp}")
except ImportError:
    print("geopandas não instalado. No Anaconda Prompt: conda install -c conda-forge geopandas")
